In [ ]:
# ============================================================
# MACHATHON 3.0 — EGYPTIAN LICENSE PLATE RECOGNITION PIPELINE
# ============================================================
# Structure:
#   Stage 1 — Plate Detection (YOLO11n)
#   Stage 2 — Character Detection + Classification (YOLO11n)
#   Stage 3 — RTL Inference + Submission Generation
# ============================================================


In [ ]:
# ─── CELL 0: Dependencies ────────────────────────────────────
!pip install ultralytics opencv-python-headless pandas numpy Pillow -q


---

## 🔧 CELL 1 — Configuration & Character Mapping


In [ ]:
import os, shutil, glob, cv2
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

# ── Paths ─────────────────────────────────────────────────────
ROOT          = Path(".")
TRAIN_IMGS    = ROOT / "train_images"
TEST_IMGS     = ROOT / "test_images"
TRAIN_CSV     = ROOT / "train_labels.csv"
TRAIN_BBXS    = ROOT / "train_bbxs"

STAGE1_LABELS = ROOT / "stage1_labels"
STAGE1_IMGS   = ROOT / "stage1_images"       # symlink / copy of train_images
STAGE2_ROOT   = ROOT / "stage2_data"          # cropped plates + char labels
STAGE2_IMGS   = STAGE2_ROOT / "images"
STAGE2_LABELS = STAGE2_ROOT / "labels"

for p in [STAGE1_LABELS, STAGE2_IMGS, STAGE2_LABELS]:
    p.mkdir(parents=True, exist_ok=True)

# ── Arabic char → class_id (0-26) ────────────────────────────
# Digits ٠-٩ → 0-9  |  17 Egyptian plate letters → 10-26
ARABIC_DIGITS  = "٠١٢٣٤٥٦٧٨٩"
ARABIC_LETTERS = "أبجدهوزحطيكلمنسعفصقرشت"[:17]   # adjust to your actual 17

CHAR2ID: dict[str, int] = {}
for i, c in enumerate(ARABIC_DIGITS):
    CHAR2ID[c] = i
for i, c in enumerate(ARABIC_LETTERS):
    CHAR2ID[c] = 10 + i

ID2CHAR = {v: k for k, v in CHAR2ID.items()}
NUM_CLASSES_STAGE2 = len(CHAR2ID)           # 27

print(f"Char vocab size: {NUM_CLASSES_STAGE2}")
print(CHAR2ID)


---

## 📦 CELL 2 — Stage 1: Plate BBox Generation (YOLO format)


In [ ]:
def parse_bbx(txt_path: Path) -> list[list[int]]:
    """Return list of [xmin,ymin,xmax,ymax] per line."""
    boxes = []
    with open(txt_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4:
                continue
            boxes.append([int(x) for x in parts[:4]])
    return boxes

def plate_box_from_chars(boxes: list[list[int]]) -> tuple[int,int,int,int]:
    """Union of all character boxes → full plate box."""
    xmins, ymins, xmaxs, ymaxs = zip(*boxes)
    return min(xmins), min(ymins), max(xmaxs), max(ymaxs)

def abs_to_yolo(xmin,ymin,xmax,ymax, W,H) -> tuple[float,...]:
    xc = ((xmin + xmax) / 2) / W
    yc = ((ymin + ymax) / 2) / H
    w  = (xmax - xmin) / W
    h  = (ymax - ymin) / H
    return xc, yc, w, h

missing_imgs = 0
for txt_path in sorted(TRAIN_BBXS.glob("*.txt")):
    img_name = txt_path.stem + ".jpg"   # adjust ext if needed
    img_path = TRAIN_IMGS / img_name

    if not img_path.exists():
        missing_imgs += 1
        continue

    img = cv2.imread(str(img_path))
    H, W = img.shape[:2]

    boxes = parse_bbx(txt_path)
    if not boxes:
        continue

    xmin, ymin, xmax, ymax = plate_box_from_chars(boxes)
    xc, yc, w, h = abs_to_yolo(xmin, ymin, xmax, ymax, W, H)

    label_path = STAGE1_LABELS / (txt_path.stem + ".txt")
    with open(label_path, "w") as f:
        f.write(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

print(f"Stage 1 labels written: {len(list(STAGE1_LABELS.glob('*.txt')))}")
print(f"Missing images skipped: {missing_imgs}")


---

## 📦 CELL 3 — Stage 1: YOLO Dataset YAML


In [ ]:
import yaml
from sklearn.model_selection import train_test_split

all_imgs = sorted(TRAIN_IMGS.glob("*.jpg"))
train_imgs, val_imgs = train_test_split(all_imgs, test_size=0.15, random_state=42)

# Write image lists
(ROOT / "s1_train.txt").write_text("\n".join(str(p.resolve()) for p in train_imgs))
(ROOT / "s1_val.txt").write_text("\n".join(str(p.resolve()) for p in val_imgs))

s1_yaml = {
    "path"  : str(ROOT.resolve()),
    "train" : "s1_train.txt",
    "val"   : "s1_val.txt",
    "nc"    : 1,
    "names" : ["plate"],
}
with open(ROOT / "stage1.yaml", "w") as f:
    yaml.dump(s1_yaml, f)

print("stage1.yaml written")


---

## 🏋️ CELL 4 — Stage 1: Train Plate Detector


In [ ]:
from ultralytics import YOLO

s1_model = YOLO("yolo11n.pt")

s1_results = s1_model.train(
    data        = str(ROOT / "stage1.yaml"),
    epochs      = 80,
    imgsz       = 640,
    batch       = 16,
    lr0         = 1e-3,
    lrf         = 0.01,
    patience    = 20,
    optimizer   = "AdamW",
    augment     = True,
    mosaic      = 1.0,
    degrees     = 5.0,
    translate   = 0.1,
    scale       = 0.3,
    fliplr      = 0.0,     # plates shouldn't be flipped LR
    flipud      = 0.0,
    project     = "runs/stage1",
    name        = "plate_det",
    exist_ok    = True,
    verbose     = False,
)

S1_BEST = Path("runs/stage1/plate_det/weights/best.pt")
print(f"Stage 1 best weights: {S1_BEST}")


---

## 📦 CELL 5 — Stage 2: Crop Plates + Build Char-Level Dataset


In [ ]:
df = pd.read_csv(TRAIN_CSV)          # columns: img_name, label
label_map = dict(zip(df["img_name"], df["label"]))

def get_char_label(img_name: str, char_idx: int) -> int:
    """
    Map character index in the bounding-box file → class_id.
    BBX rows are stored left→right visually; label string is RTL.
    So char_idx 0 in bbx = rightmost char in Arabic string (index -1).
    """
    label = label_map.get(img_name, "")
    # Reverse label so index 0 = rightmost Arabic char
    rev_label = label[::-1]
    if char_idx >= len(rev_label):
        return -1   # unknown / skip
    char = rev_label[char_idx]
    return CHAR2ID.get(char, -1)

crop_count = 0
for txt_path in sorted(TRAIN_BBXS.glob("*.txt")):
    img_name = txt_path.stem + ".jpg"
    img_path = TRAIN_IMGS / img_name
    if not img_path.exists():
        continue

    img = cv2.imread(str(img_path))
    H, W = img.shape[:2]
    boxes = parse_bbx(txt_path)
    if not boxes:
        continue

    # ── Crop full plate ───────────────────────────────────────
    px1, py1, px2, py2 = plate_box_from_chars(boxes)
    px1, py1 = max(0,px1-4), max(0,py1-4)   # small padding
    px2, py2 = min(W,px2+4), min(H,py2+4)
    plate_crop = img[py1:py2, px1:px2]
    PH, PW = plate_crop.shape[:2]

    stem = txt_path.stem
    crop_path = STAGE2_IMGS / f"{stem}.jpg"
    cv2.imwrite(str(crop_path), plate_crop)

    # ── Write char YOLO labels relative to cropped plate ──────
    label_lines = []
    for idx, (xmin,ymin,xmax,ymax) in enumerate(boxes):
        class_id = get_char_label(img_name, idx)
        if class_id < 0:
            continue
        # Shift coords to plate-crop space
        xmin_c = xmin - px1; xmax_c = xmax - px1
        ymin_c = ymin - py1; ymax_c = ymax - py1
        xc, yc, bw, bh = abs_to_yolo(xmin_c, ymin_c, xmax_c, ymax_c, PW, PH)
        label_lines.append(f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

    if label_lines:
        (STAGE2_LABELS / f"{stem}.txt").write_text("\n".join(label_lines))
        crop_count += 1

print(f"Stage 2 crops+labels: {crop_count}")


---

## 📦 CELL 6 — Stage 2: Dataset YAML


In [ ]:
all_crops = sorted(STAGE2_IMGS.glob("*.jpg"))
s2_train, s2_val = train_test_split(all_crops, test_size=0.15, random_state=42)

(ROOT / "s2_train.txt").write_text("\n".join(str(p.resolve()) for p in s2_train))
(ROOT / "s2_val.txt").write_text("\n".join(str(p.resolve()) for p in s2_val))

s2_yaml = {
    "path"  : str(ROOT.resolve()),
    "train" : "s2_train.txt",
    "val"   : "s2_val.txt",
    "nc"    : NUM_CLASSES_STAGE2,
    "names" : [ID2CHAR[i] for i in range(NUM_CLASSES_STAGE2)],
}
with open(ROOT / "stage2.yaml", "w") as f:
    yaml.dump(s2_yaml, f, allow_unicode=True)

print("stage2.yaml written")


---

## 🏋️ CELL 7 — Stage 2: Train Character Detector


In [ ]:
s2_model = YOLO("yolo11n.pt")

s2_results = s2_model.train(
    data        = str(ROOT / "stage2.yaml"),
    epochs      = 100,
    imgsz       = 320,          # plates are wide/short → smaller imgsz OK
    batch       = 32,
    lr0         = 1e-3,
    lrf         = 0.01,
    patience    = 25,
    optimizer   = "AdamW",
    augment     = True,
    mosaic      = 0.5,
    degrees     = 3.0,
    translate   = 0.05,
    scale       = 0.2,
    fliplr      = 0.0,
    flipud      = 0.0,
    project     = "runs/stage2",
    name        = "char_det",
    exist_ok    = True,
    verbose     = False,
)

S2_BEST = Path("runs/stage2/char_det/weights/best.pt")
print(f"Stage 2 best weights: {S2_BEST}")


---

## 🔍 CELL 8 — Stage 3: Inference Pipeline (RTL Sorting)


In [ ]:
from ultralytics import YOLO
import numpy as np
import cv2

# ── Load models (update paths post-training) ─────────────────
S1_BEST = Path("runs/stage1/plate_det/weights/best.pt")
S2_BEST = Path("runs/stage2/char_det/weights/best.pt")

s1_infer = YOLO(str(S1_BEST))
s2_infer = YOLO(str(S2_BEST))

# ── RTL Line-Aware Sort ───────────────────────────────────────
def sort_rtl(detections: list[dict], line_gap_ratio: float = 0.6) -> str:
    """
    detections: list of {"class_id":int, "xc":float, "yc":float, "h":float}
    Returns Arabic string sorted right-to-left, multi-line aware.
    """
    if not detections:
        return ""

    # Group by line: cluster on y_center using median char height as threshold
    detections = sorted(detections, key=lambda d: d["yc"])
    med_h = np.median([d["h"] for d in detections])
    thresh = med_h * line_gap_ratio

    lines: list[list[dict]] = []
    current_line = [detections[0]]
    for det in detections[1:]:
        if abs(det["yc"] - np.mean([d["yc"] for d in current_line])) < thresh:
            current_line.append(det)
        else:
            lines.append(current_line)
            current_line = [det]
    lines.append(current_line)

    # Sort lines top→bottom, within each line sort right→left (desc xc)
    result_chars = []
    for line in lines:
        sorted_line = sorted(line, key=lambda d: d["xc"], reverse=True)
        result_chars.extend(ID2CHAR.get(d["class_id"], "?") for d in sorted_line)

    return "".join(result_chars)


def predict_plate(img_path: str,
                  s1_conf: float = 0.4,
                  s2_conf: float = 0.35,
                  pad: int = 6) -> str:
    """
    End-to-end: raw image → Arabic plate string.
    Returns empty string if no plate / chars detected.
    """
    img = cv2.imread(img_path)
    if img is None:
        return ""
    H, W = img.shape[:2]

    # ── Stage 1: detect plate ─────────────────────────────────
    s1_res = s1_infer(img, conf=s1_conf, verbose=False)[0]
    if len(s1_res.boxes) == 0:
        # Fallback: treat full image as plate
        plate_crop = img
    else:
        # Pick highest-confidence plate box
        box = s1_res.boxes[s1_res.boxes.conf.argmax()]
        x1,y1,x2,y2 = box.xyxy[0].cpu().numpy().astype(int)
        x1,y1 = max(0,x1-pad), max(0,y1-pad)
        x2,y2 = min(W,x2+pad), min(H,y2+pad)
        plate_crop = img[y1:y2, x1:x2]

    PH, PW = plate_crop.shape[:2]
    if PH == 0 or PW == 0:
        return ""

    # ── Stage 2: detect + classify characters ─────────────────
    s2_res = s2_infer(plate_crop, conf=s2_conf, verbose=False)[0]
    if len(s2_res.boxes) == 0:
        return ""

    detections = []
    for box in s2_res.boxes:
        cls_id = int(box.cls[0].item())
        x1b,y1b,x2b,y2b = box.xyxy[0].cpu().numpy()
        xc = ((x1b + x2b) / 2) / PW
        yc = ((y1b + y2b) / 2) / PH
        h  = (y2b - y1b) / PH
        detections.append({"class_id": cls_id, "xc": xc, "yc": yc, "h": h})

    return sort_rtl(detections)


---

## 🧪 CELL 9 — Quick Sanity Check (Train Set Sample)


In [ ]:
from tqdm.auto import tqdm

df_train = pd.read_csv(TRAIN_CSV)
sample = df_train.sample(10, random_state=0)

rows = []
for _, row in sample.iterrows():
    img_path = str(TRAIN_IMGS / row["img_name"])
    pred = predict_plate(img_path)
    rows.append({"img_name": row["img_name"],
                 "gt": row["label"],
                 "pred": pred,
                 "match": pred == row["label"]})

df_check = pd.DataFrame(rows)
print(df_check)
print(f"\nSample accuracy: {df_check['match'].mean():.1%}")


---

## 📊 CELL 10 — Full Validation Evaluation (CER + Accuracy)


In [ ]:
def cer(gt: str, pred: str) -> float:
    """Character Error Rate via Levenshtein distance."""
    import Levenshtein as lev
    if len(gt) == 0:
        return 0.0 if len(pred) == 0 else 1.0
    return lev.distance(gt, pred) / len(gt)

try:
    import Levenshtein
    HAS_LEV = True
except ImportError:
    !pip install python-Levenshtein -q
    import Levenshtein
    HAS_LEV = True

val_results = []
for img_name in tqdm([p.name for p in val_imgs], desc="Validating"):
    gt    = label_map.get(img_name, "")
    pred  = predict_plate(str(TRAIN_IMGS / img_name))
    val_results.append({
        "img_name": img_name,
        "gt"      : gt,
        "pred"    : pred,
        "exact"   : int(pred == gt),
        "cer"     : cer(gt, pred),
    })

df_val = pd.DataFrame(val_results)
print(f"Exact Match Accuracy : {df_val['exact'].mean():.4%}")
print(f"Mean CER             : {df_val['cer'].mean():.4f}")
df_val[df_val["exact"] == 0].head(10)


---

## 🚀 CELL 11 — Generate Test Submission


In [ ]:
test_img_paths = sorted(TEST_IMGS.glob("*.jpg"))

submission_rows = []
for img_path in tqdm(test_img_paths, desc="Inference"):
    pred = predict_plate(str(img_path))
    submission_rows.append({
        "img_name": img_path.name,
        "label"   : pred,
    })

df_submission = pd.DataFrame(submission_rows)
df_submission.to_csv("submission.csv", index=False, encoding="utf-8-sig")
print(f"submission.csv saved — {len(df_submission)} rows")
df_submission.head(10)


---

## ♻️ CELL 12 — Optional: TTA (Test-Time Augmentation) Wrapper


In [ ]:
def predict_plate_tta(img_path: str,
                      scales: list[float] = [320, 480, 640]) -> str:
    """
    Run inference at multiple resolutions, pick majority vote per char position.
    Simple TTA: vote on full string prediction across scales.
    """
    from collections import Counter
    preds = []
    img_orig = cv2.imread(img_path)
    if img_orig is None:
        return ""

    for scale in scales:
        # Resize to square keeping aspect, run pipeline on resized image
        h, w = img_orig.shape[:2]
        ratio = scale / max(h, w)
        resized = cv2.resize(img_orig, (int(w*ratio), int(h*ratio)))
        tmp = "/tmp/_tta_tmp.jpg"
        cv2.imwrite(tmp, resized)
        preds.append(predict_plate(tmp))

    # Return most common prediction
    return Counter(preds).most_common(1)[0][0]

# Uncomment to use TTA on submission:
# submission_rows = []
# for img_path in tqdm(test_img_paths, desc="TTA Inference"):
#     pred = predict_plate_tta(str(img_path))
#     submission_rows.append({"img_name": img_path.name, "label": pred})
# df_submission = pd.DataFrame(submission_rows)
# df_submission.to_csv("submission_tta.csv", index=False, encoding="utf-8-sig")


---

## 📋 CELL 13 — Char Mapping Reference (Verify & Customize)


In [ ]:
# ── CRITICAL: Verify this against your actual dataset ─────────
# Run this cell to audit what characters actually appear in train_labels.csv
df_labels = pd.read_csv(TRAIN_CSV)
all_chars  = set("".join(df_labels["label"].astype(str).tolist()))

print(f"Unique chars in dataset: {len(all_chars)}")
print(sorted(all_chars))

# Re-build CHAR2ID from actual data (recommended over hardcoded map)
sorted_chars = sorted(all_chars)
CHAR2ID_ACTUAL = {c: i for i, c in enumerate(sorted_chars)}
ID2CHAR_ACTUAL = {v: k for k, v in CHAR2ID_ACTUAL.items()}
print(f"\nAuto-built vocab ({len(CHAR2ID_ACTUAL)} classes):")
print(CHAR2ID_ACTUAL)

# ⚠ If using auto-built map, re-run Cells 5-7 with CHAR2ID = CHAR2ID_ACTUAL


---

## 📌 Pipeline Summary

```
train_bbxs/*.txt
       │
       ├──[CELL 2]──► stage1_labels/     (plate YOLO boxes)
       │                    │
       │               [CELL 4] YOLO11n train ──► runs/stage1/best.pt
       │
       └──[CELL 5]──► stage2_data/       (cropped plates + char YOLO boxes)
                            │
                       [CELL 7] YOLO11n train ──► runs/stage2/best.pt
                            │
                       [CELL 8] predict_plate()
                         ├─ Stage1: detect & crop plate
                         ├─ Stage2: detect & classify chars
                         └─ sort_rtl(): RTL + multiline sort
                            │
                       [CELL 11] ──► submission.csv
```

> **Key tuning levers:**
> - Increase `epochs` to 150+ and use `yolo11s.pt` (small) for better accuracy at modest speed cost.
> - Tune `line_gap_ratio` in `sort_rtl()` if dual-line plates are misread.
> - Use `CHAR2ID_ACTUAL` from Cell 13 to guarantee the vocabulary matches your exact dataset — don't rely on the hardcoded map.
> - For plates with no Stage 1 detection, the fallback in `predict_plate()` uses the full image — consider a second-pass with lower `s1_conf=0.25`.
